# inference

> Turn a trained model into predictions: probability maps (optionally test-time augmented), label decoding, and an operating-threshold search for binary tasks.

`predict` returns per-class probabilities for both binary (sigmoid) and multi-class (softmax). `decode` collapses them to a label mask. `find_operating_threshold` sweeps the binary decision threshold to hit a target recall, which matters when a miss is far costlier than a false alarm.

In [ ]:
#| default_exp inference

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Optional, Sequence

import numpy as np
import torch


In [ ]:
#| exporti
from segmentation_tools_pt.metrics import ObjectMetrics

## Predict probabilities

One forward pass returns `(C, H, W)` probabilities. With `tta=True` we average predictions over the four 90-degree rotations, rotating each prediction back before averaging. TTA trades compute for a small, usually-positive accuracy bump on thin structures.

`task` disambiguates `C > 1`: `multiclass` (default) uses softmax -- classes are mutually exclusive, so probabilities sum to 1 across channels. `multilabel` uses an independent sigmoid per channel instead, so overlapping classes (e.g. `void` inside `chip`) aren't forced to compete for probability mass. `binary` (`C == 1`) always uses sigmoid regardless of `task`.

In [ ]:
#| export
@torch.no_grad()
def predict(
    model,                 # trained model returning (B, C, H, W) logits
    image: torch.Tensor,   # a single (C, H, W) input
    device: torch.device,  # cpu or cuda
    tta: bool = False,     # average over 90-degree rotations
    n_augs: int = 4,       # number of rotations when tta=True
    activate: bool = True, # apply sigmoid/softmax to logits
    task: str = None,      # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities: sigmoid if `C==1` or `task='multilabel'`, else softmax."""
    model.eval()
    x = image.unsqueeze(0).to(device)

    def forward(t: torch.Tensor) -> torch.Tensor:
        logits = model(t)
        if not activate:
            return logits
        if logits.shape[1] == 1 or task == 'multilabel':
            return torch.sigmoid(logits)
        return torch.softmax(logits, dim=1)

    if not tta:
        return forward(x).squeeze(0)
    preds = []
    for k in range(n_augs):
        pr = forward(torch.rot90(x, k, dims=[2, 3]))
        preds.append(torch.rot90(pr, -k, dims=[2, 3]))
    return torch.stack(preds).mean(0).squeeze(0)

In [ ]:
#| export
def decode(
    probs: torch.Tensor,     # (C, H, W) probabilities
    threshold=0.5,            # decision threshold: float (binary/multilabel-shared), or per-channel Sequence[float] for multilabel
    task: str = None,         # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Collapse probabilities to a label mask.

    `binary`/`multiclass` (default behavior, unchanged) return `(H, W)`: a
    thresholded `{0,1}` mask for `C==1`, or an argmax label map for `C>1`.
    `multilabel` returns `(C, H, W)` -- classes can overlap, so there is no
    single `(H,W)` label map to collapse to; each channel is thresholded
    independently instead.

    Example:
        >>> decode(torch.tensor([[[0.2, 0.9]]])).tolist()
        [[0, 1]]
    """
    if task == 'multilabel':
        c = probs.shape[0]
        # duck-type "sequence of per-channel thresholds" rather than isinstance(list, tuple):
        # Hydra/OmegaConf configs hand back ListConfig, not a plain list, for a YAML list.
        is_seq = hasattr(threshold, '__len__') and not isinstance(threshold, (str, bytes))
        t = list(threshold) if is_seq else [threshold] * c
        return torch.stack([probs[k] >= t[k] for k in range(c)]).long()
    if probs.shape[0] == 1:
        return (probs[0] >= threshold).long()
    return probs.argmax(0)

In [ ]:
#| test
# test: shapes for binary and multi-class, with and without TTA
import torch.nn as nn
dev = torch.device('cpu')
mb = nn.Conv2d(3, 1, 3, padding=1)
p = predict(mb, torch.rand(3, 16, 16), dev)
assert p.shape == (1, 16, 16)
assert predict(mb, torch.rand(3, 16, 16), dev, tta=True).shape == (1, 16, 16)
assert decode(p).shape == (16, 16)
mc = nn.Conv2d(3, 3, 3, padding=1)
pc = predict(mc, torch.rand(3, 16, 16), dev)
assert pc.shape == (3, 16, 16) and decode(pc).shape == (16, 16)

In [ ]:
#| test
# test: multilabel predict/decode -- independent per-channel sigmoid, no softmax, overlap preserved
m3 = nn.Conv2d(3, 3, 3, padding=1)
p3 = predict(m3, torch.rand(3, 16, 16), dev, task='multilabel')
assert p3.shape == (3, 16, 16)
assert not torch.allclose(p3.sum(0), torch.ones(16, 16))  # not softmax-normalised

d3 = decode(p3, task='multilabel')
assert d3.shape == (3, 16, 16)  # unlike multiclass decode, NOT collapsed to (H,W)

# per-channel thresholds actually take effect
probs_fixed = torch.full((3, 4, 4), 0.6)
d_low = decode(probs_fixed, threshold=0.5, task='multilabel')
d_high = decode(probs_fixed, threshold=[0.5, 0.7, 0.5], task='multilabel')
assert bool(d_low[1].all()) and not bool(d_high[1].any())  # channel 1: 0.6>=0.5 fires, 0.6>=0.7 doesn't
print('multilabel predict/decode ✓  independent sigmoid, (C,H,W) decode, per-channel thresholds')

## Hierarchy post-process (multilabel)

`decode(task='multilabel')` thresholds each channel independently, so a
trained model can still predict a `void`/`lifted` pixel outside its `chip`
footprint even with `losses.hierarchy_regularizer_loss` nudging it during
training (that's a soft penalty, not a guarantee). `apply_hierarchy` hard-
enforces the containment relationship at inference time: any sub-channel
pixel not also covered by its super-channel is zeroed out.

In [ ]:
#| export
def apply_hierarchy(
    masks: torch.Tensor,  # (C, H, W) or (B, C, H, W) decoded/binary multilabel masks
    pairs,                  # [(sub_idx, super_idx), ...], e.g. [(1, 0), (2, 0)] for (void,chip),(lifted,chip)
) -> torch.Tensor:
    """Zero out sub-channel pixels not covered by their super-channel. Returns a new tensor."""
    if not pairs:
        return masks
    masks = masks.clone()
    c_dim = -3  # channel is always third-from-last: (...,C,H,W)
    for s, p in pairs:
        sub = masks.select(c_dim, s)
        sup = masks.select(c_dim, p)
        if masks.dtype == torch.bool:
            sub &= sup
        else:
            sub.mul_(sup)
    return masks

In [ ]:
#| test
# test: apply_hierarchy zeroes sub-channel pixels outside their super-channel, leaves the rest alone
chip = torch.zeros(4, 4); chip[1:3, 1:3] = 1
void = torch.zeros(4, 4); void[0:2, 0:2] = 1  # only (1,1) is inside chip; (0,0),(0,1),(1,0) leak outside
masks = torch.stack([chip, void]).long()  # (C=2, H, W): 0=chip, 1=void

fixed = apply_hierarchy(masks, pairs=[(1, 0)])
assert fixed[1, 1, 1] == 1        # inside chip -> kept
assert fixed[1, 0, 0] == 0        # outside chip -> zeroed
assert torch.equal(fixed[0], masks[0])  # super-channel (chip) itself is untouched
assert masks[1, 0, 0] == 1        # original input not mutated (clone, not in-place on the caller's tensor)

# batched (B, C, H, W) works the same way
batched = masks.unsqueeze(0).repeat(2, 1, 1, 1)
fixed_b = apply_hierarchy(batched, pairs=[(1, 0)])
assert fixed_b.shape == (2, 2, 4, 4)
assert bool((fixed_b[:, 1, 0, 0] == 0).all())

# empty pairs -> no-op, same tensor object back (cheap default)
assert apply_hierarchy(masks, []) is masks

# also works on a bool mask (as produced by decode(..., task='multilabel'))
bool_masks = masks.bool()
fixed_bool = apply_hierarchy(bool_masks, [(1, 0)])
assert fixed_bool.dtype == torch.bool and bool(fixed_bool[1, 0, 0]) is False
print('apply_hierarchy ✓  hard-enforces containment, (C,H,W) + (B,C,H,W) + bool, no input mutation')

## Operating-threshold search (binary)

Sweep the sigmoid threshold over a validation loader and report pixel recall/precision/IoU plus object-level detection at each. We recommend the **highest** threshold that still meets `target_recall`: highest precision among the options that do not start missing targets.

In [ ]:
#| export
@torch.no_grad()
def find_operating_threshold(
    model,                          # trained binary model
    val_loader,                     # validation DataLoader of {'image','mask'}
    device: torch.device,           # cpu or cuda
    target_recall: float = 0.99,    # minimum pixel recall to accept
    overlap_thresh: float = 0.5,    # object overlap fraction for detection
    thresholds: Optional[Sequence[float]] = None,  # thresholds to scan
) -> Dict:
    """Scan thresholds; return the recommended one plus the full sweep."""
    model.eval()
    thresholds = list(thresholds) if thresholds is not None else list(np.arange(0.05, 0.95, 0.05))
    prob_imgs, gt_imgs = [], []
    for batch in val_loader:
        images = batch['image'].to(device)
        p = torch.sigmoid(model(images))[:, 0].cpu().numpy()
        prob_imgs.append(p)
        gt_imgs.append((batch['mask'].numpy() > 0).astype(np.uint8))
    probs = np.concatenate(prob_imgs)        # (N, H, W)
    gts = np.concatenate(gt_imgs)            # (N, H, W)
    pf, gf = probs.reshape(-1), gts.reshape(-1).astype(bool)
    sweep = []
    for t in thresholds:
        pred = pf >= t
        tp = int((pred & gf).sum()); fp = int((pred & ~gf).sum()); fn = int((~pred & gf).sum())
        obj = ObjectMetrics(overlap_thresh)
        for pimg, gimg in zip(probs, gts):
            obj.update((pimg >= t).astype(np.uint8), gimg)
        od = obj.compute()
        sweep.append({'threshold': round(float(t), 2),
                      'recall': round(tp / (tp + fn + 1e-9), 4),
                      'precision': round(tp / (tp + fp + 1e-9), 4),
                      'iou': round(tp / (tp + fp + fn + 1e-9), 4),
                      'detection_rate': round(od['detection_rate'], 4),
                      'missed': od['missed'], 'false_alarms': od['false_alarms']})
    ok = [r for r in sweep if r['recall'] >= target_recall]
    best = max(ok, key=lambda r: r['threshold']) if ok else max(sweep, key=lambda r: r['recall'])
    return {'recommended_threshold': best['threshold'], 'achieved_recall': best['recall'],
            'achieved_precision': best['precision'], 'sweep': sweep}

In [ ]:
#| test
# test: threshold search returns a valid recommendation on a tiny val loader
from torch.utils.data import Dataset, DataLoader

class _Val(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:12, 4:12] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

res = find_operating_threshold(mb, DataLoader(_Val(), batch_size=2), dev, target_recall=0.0, thresholds=[0.3, 0.5, 0.7])
assert 0.0 < res['recommended_threshold'] < 1.0 and len(res['sweep']) == 3

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()